In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח באמצעות הדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
בנוסף ל[ויזואליזציה של הוראות על Circuit](/guides/visualize-circuits), ייתכן שתרצה לבצע ויזואליזציה של תזמון ה-Circuit באמצעות מתודת Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). ויזואליזציה זו יכולה לעזור לך לאתר במהירות זמן סרק על Qubit-ים, למשל. עם זאת, מתודה זו אינה מחזירה תוצאות מדויקות עבור Circuits דינמיים.  לביצוע ויזואליזציה של תזמון Circuit דינמי, השתמש במתודת `draw_circuit_schedule_timing`, כפי שמתואר בסעיף [תמיכת Qiskit Runtime](#qr-support).

## דוגמאות

כדי לבצע ויזואליזציה של תוכנית Circuit מתוזמנת, אפשר לקרוא לפונקציה זו עם קבוצת ארגומנטי בקרה. ניתן לשנות את מרבית מראה תמונת הפלט באמצעות גיליון סגנון, אך זה אינו חובה.

### ציור עם גיליון הסגנון ברירת המחדל

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![פלט תא הקוד הקודם](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### ציור עם גיליון סגנון מתאים לניפוי באגים בתוכנית

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![פלט תא הקוד הקודם](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

אפשר ליצור פונקציות generator או layout מותאמות אישית ולעדכן גיליון סגנון קיים עם הפונקציות המותאמות. כך ניתן לשלוט ברוב מראה תמונת הפלט מבלי לשנות את בסיס הקוד של מצייר ה-Circuit המתוזמן. ראה את מסמך ה-API של [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) לדוגמאות נוספות.
<span id="qr-support"></span>
## תמיכת Qiskit Runtime
בעוד שמצייר ציר הזמן המובנה ב-Qiskit שימושי עבור Circuits סטטיים, ייתכן שהוא לא ישקף במדויק את התזמון של [Circuits דינמיים](/guides/classical-feedforward-and-control-flow) בשל פעולות משתמעות כמו broadcasting ו-branch determination. כחלק מתמיכה ב-Circuits דינמיים, Qiskit Runtime מחזיר את מידע תזמון ה-Circuit המדויק בתוך תוצאות ה-job כאשר מתבקש.

> **Note:** - זו פונקציה ניסיונית. היא נמצאת בסטטוס שחרור תצוגה מקדימה ולכן עשויה להשתנות.
> - פונקציה זו חלה רק על jobs של Qiskit Runtime Sampler.
> - למרות שזמן ה-Circuit הכולל מוחזר ב-metadata של "compilation", זה אינו הזמן המשמש לחיוב (זמן קוונטי).
### הפעלת אחזור נתוני תזמון
כדי להפעיל אחזור נתוני תזמון, הגדר את הדגל הניסיוני `scheduler_timing` ל-`True` בעת הרצת ה-primitive job.